# The Stories We Carry — Kaggle GPU Production Notebook

Runs on **Kaggle 2× T4** (free weekly GPU quota) to generate:
1. **Moving Shots** using **Wan 2.2 TI2V-5B** (`Wan-AI/Wan2.2-TI2V-5B`, Apache 2.0) from `assets/images/` keyframes.
2. **Original Score Candidates** using **ACE-Step 1.5** (`ACE-Step/ACE-Step-1.5`, MIT) with Ethiopian tizita-inspired instrumental cues.

Every model call logs latency, parameters, seed, and GPU minutes to `runs.jsonl`, and saves outputs after each shot so session timeouts never lose completed work.

In [ ]:
# 1. Setup: clone repo or point PROJECT_ROOT to your uploaded dataset/repo
import json, os, time
from pathlib import Path
from datetime import datetime, timezone

PROJECT_ROOT = Path("/kaggle/working/the-stories-we-carry")
# If not cloned yet:
# !git clone <YOUR_REPO_URL> /kaggle/working/the-stories-we-carry

STORYBOARD_PATH = PROJECT_ROOT / "storyboard.json"
VIDEO_OUT_DIR = PROJECT_ROOT / "outputs" / "video"
MUSIC_OUT_DIR = PROJECT_ROOT / "assets" / "music"
RUNS_LOG = PROJECT_ROOT / "runs.jsonl"

VIDEO_OUT_DIR.mkdir(parents=True, exist_ok=True)
MUSIC_OUT_DIR.mkdir(parents=True, exist_ok=True)

def log_gpu_run(stage, scene_id, model, params, seed, latency_s, output_path):
    record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "stage": stage,
        "scene_id": scene_id,
        "model": model,
        "parameters": params,
        "seed": seed,
        "latency_seconds": round(latency_s, 3),
        "gpu_minutes": round(latency_s / 60.0, 3),
        "estimated_cost_usd": 0.0,
        "status": "success",
        "output_path": str(output_path),
    }
    with RUNS_LOG.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

In [ ]:
# 2. Install Wan 2.2 / Diffusers & ACE-Step dependencies
!pip install -q --upgrade diffusers transformers accelerate imageio imageio-ffmpeg ftfy
!pip install -q git+https://github.com/ACE-Step/ACE-Step-1.5.git

In [ ]:
# 3. Generate Moving Shots with Wan 2.2 TI2V-5B (drafting at 480p, 24fps)
import torch
from PIL import Image
from diffusers import WanImageToVideoPipeline
from diffusers.utils import export_to_video

storyboard = json.loads(STORYBOARD_PATH.read_text(encoding="utf-8"))
MODEL_ID = "Wan-AI/Wan2.2-TI2V-5B"
SEED = 1001
NUM_FRAMES = 121  # ~5s at 24fps (assembly loops/extends cleanly to 12.5s slot)

pipe = WanImageToVideoPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()

MOTION_PROMPTS = {
    "scene-01": "Gentle morning sunlight dust motes drifting across the sepia photograph, subtle slow push-in camera motion, serene and nostalgic.",
    "scene-02": "Elderly hands gently pouring steaming coffee from the clay jebena, rising steam, soft subtle camera drift.",
    "scene-03": "Soft breeze moving the woven Ethiopian fabric slightly, warm light shifting gently, slow locked-off breathing camera.",
    "scene-04": "Golden mist drifting slowly across the Ethiopian mountain valley at sunrise, gentle wide pan.",
    "scene-05": "Warm candlelight flickering softly over the handwritten Amharic letter inside the wooden box, slow intimate push-in.",
    "scene-06": "Golden sunrise rays gradually brightening over the ancient Ethiopian village, hopeful slow upward tilt.",
}

for scene in storyboard["scenes"]:
    sid = scene["id"]
    out_mp4 = VIDEO_OUT_DIR / f"{sid}.mp4"
    if out_mp4.is_file():
        print(f"[SKIP] {sid} already generated.")
        continue

    img_path = PROJECT_ROOT / scene["visual"]["image_path"]
    image = Image.open(img_path).convert("RGB").resize((480, 832))
    prompt = MOTION_PROMPTS.get(sid, "Slow gentle cinematic camera push-in, subtle natural motion.")

    t0 = time.monotonic()
    generator = torch.Generator(device="cuda").manual_seed(SEED)
    frames = pipe(
        image=image,
        prompt=prompt,
        negative_prompt="jitter, morphing, distorted text, fast camera shake, blurry faces",
        num_frames=NUM_FRAMES,
        num_inference_steps=30,
        guidance_scale=5.0,
        generator=generator,
    ).frames[0]
    export_to_video(frames, str(out_mp4), fps=24)
    latency = time.monotonic() - t0
    log_gpu_run("moving_shots", sid, MODEL_ID, {"res": "480x832", "frames": NUM_FRAMES, "steps": 30}, SEED, latency, out_mp4.relative_to(PROJECT_ROOT))
    print(f"[OK] {sid} saved in {latency:.1f}s")

In [ ]:
# 4. Generate 3 Original Score Candidates with ACE-Step 1.5 (instrumental)
from acestep.pipeline_ace_step import ACEStepPipeline

ace = ACEStepPipeline(dtype="bfloat16")
caption = "warm contemplative Ethiopian acoustic instrumental, krar and washint flute texture, gentle strings, nostalgic cinematic underscore, no vocals, instrumental"

for idx, seed in enumerate([42, 108, 777], start=1):
    out_wav = MUSIC_OUT_DIR / f"candidate_{idx}.wav"
    t0 = time.monotonic()
    ace(
        audio_duration=75.0,
        prompt=caption,
        lyrics="[instrumental]",
        infer_step=27,
        guidance_scale=7.0,
        manual_seeds=[seed],
        save_path=str(out_wav),
    )
    latency = time.monotonic() - t0
    log_gpu_run("music_candidate", None, "ACE-Step/ACE-Step-1.5", {"duration": 75.0, "steps": 27}, seed, latency, out_wav.relative_to(PROJECT_ROOT))
    print(f"[OK] Candidate {idx} saved in {latency:.1f}s")

In [ ]:
# 5. Package outputs to download back to your local machine
!cd /kaggle/working/the-stories-we-carry && zip -r /kaggle/working/kaggle_outputs.zip outputs/video assets/music runs.jsonl
print("Download /kaggle/working/kaggle_outputs.zip and extract into your local repo root.")